In [1]:
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, Dataset
from transformers import AutoTokenizer, AutoModel, get_linear_schedule_with_warmup
from sklearn.metrics import accuracy_score, recall_score, precision_score, f1_score
from sklearn.model_selection import train_test_split
from tqdm import tqdm
import random

In [2]:
def seed_everything(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

seed_everything()

In [3]:
class Config:
    CLS = [0]  # XLM-RoBERTa CLS token
    SEP = [2]  # XLM-RoBERTa SEP token
    VALUE_TOKEN = [1]  # XLM-RoBERTa padding token
    MAX_LEN = 128
    TRAIN_BATCH_SIZE = 16  # Reducing batch size for better performance
    VAL_BATCH_SIZE = 8
    EPOCHS = 15
    TOKENIZER = AutoTokenizer.from_pretrained("xlm-roberta-base")

tokenizer_config.json:   0%|          | 0.00/25.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/615 [00:00<?, ?B/s]

sentencepiece.bpe.model:   0%|          | 0.00/5.07M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/9.10M [00:00<?, ?B/s]

In [4]:
class CustomDataset(Dataset):
    def __init__(self, texts, pos_tags, ner_tags):
        self.texts = texts
        self.pos_tags = pos_tags
        self.ner_tags = ner_tags
  
    def __len__(self):
        return len(self.texts)

    def __getitem__(self, index):
        texts = self.texts[index]
        pos_tags = self.pos_tags[index]
        ner_tags = self.ner_tags[index]
    
        ids, target_pos, target_ner = [], [], []

        for i, s in enumerate(texts):
            inputs = Config.TOKENIZER.encode(s, add_special_tokens=False)
            input_len = len(inputs)
            ids.extend(inputs)
            target_pos.extend(input_len * [pos_tags[i]])
            target_ner.extend(input_len * [ner_tags[i]])
        
        ids = ids[:Config.MAX_LEN - 2]
        target_pos = target_pos[:Config.MAX_LEN - 2]
        target_ner = target_ner[:Config.MAX_LEN - 2]

        ids = Config.CLS + ids + Config.SEP
        target_pos = Config.VALUE_TOKEN + target_pos + Config.VALUE_TOKEN
        target_ner = Config.VALUE_TOKEN + target_ner + Config.VALUE_TOKEN

        mask = [1] * len(ids)
        token_type_ids = [0] * len(ids)

        padding_len = Config.MAX_LEN - len(ids)
        ids = ids + ([1] * padding_len)  # Padding token for XLM-RoBERTa
        target_pos = target_pos + ([1] * padding_len)
        target_ner = target_ner + ([1] * padding_len)
        mask = mask + ([0] * padding_len)
        token_type_ids = token_type_ids + ([0] * padding_len)

        return {
            "ids": torch.tensor(ids, dtype=torch.long),
            "mask": torch.tensor(mask, dtype=torch.long),
            "token_type_ids": torch.tensor(token_type_ids, dtype=torch.long),
            "target_pos": torch.tensor(target_pos, dtype=torch.long),
            "target_ner": torch.tensor(target_ner, dtype=torch.long)
        }

In [5]:
class NERPOSModel(nn.Module):
    def __init__(self, num_pos, num_ner):
        super(NERPOSModel, self).__init__()
        self.num_pos = num_pos
        self.num_ner = num_ner
        self.bert = AutoModel.from_pretrained("xlm-roberta-base")
        
        # Unfreeze only the last few layers of the model
        for param in self.bert.parameters():
            param.requires_grad = False
        for param in self.bert.encoder.layer[-4:].parameters():
            param.requires_grad = True
        
        self.bert_drop = nn.Dropout(0.3)
        self.out_pos = nn.Linear(768, self.num_pos)
        self.out_ner = nn.Linear(768, self.num_ner)
        
    def forward(self, ids, mask, token_type_ids, target_pos=None, target_ner=None):
        output = self.bert(ids, attention_mask=mask)
        bert_out = self.bert_drop(output.last_hidden_state)
        pos = self.out_pos(bert_out)
        ner = self.out_ner(bert_out)
        
        loss = None
        if target_pos is not None and target_ner is not None:
            criterion = nn.CrossEntropyLoss()
            active_loss = mask.view(-1) == 1
            active_logits_pos = pos.view(-1, self.num_pos)
            active_logits_ner = ner.view(-1, self.num_ner)
            active_labels_pos = torch.where(active_loss, target_pos.view(-1), torch.tensor(criterion.ignore_index).type_as(target_pos))
            active_labels_ner = torch.where(active_loss, target_ner.view(-1), torch.tensor(criterion.ignore_index).type_as(target_ner))
            loss_pos = criterion(active_logits_pos, active_labels_pos)
            loss_ner = criterion(active_logits_ner, active_labels_ner)
            loss = (loss_pos + loss_ner) / 2

        return pos, ner, loss

In [6]:
def parse_dataset(file_path):
    sentences, words, pos_tags, ner_tags = [], [], [], []
    with open(file_path, 'r', encoding='utf-8') as file:
        current_sentence, current_pos, current_ner = [], [], []
        for line in file:
            line = line.strip()
            if line:
                parts = line.split('\t')
                if len(parts) == 3:
                    token, pos, ner = parts
                    current_sentence.append(token)
                    current_pos.append(pos)
                    current_ner.append(ner)
            else:
                if current_sentence:
                    sentences.append(" ".join(current_sentence))
                    words.append(current_sentence)
                    pos_tags.append(current_pos)
                    ner_tags.append(current_ner)
                    current_sentence, current_pos, current_ner = [], [], []
        if current_sentence:
            sentences.append(" ".join(current_sentence))
            words.append(current_sentence)
            pos_tags.append(current_pos)
            ner_tags.append(current_ner)

    df = pd.DataFrame({'words': words, 'pos': pos_tags, 'ner': ner_tags})

    pos_set = set(tag for sublist in df['pos'] for tag in sublist)
    ner_set = set(tag for sublist in df['ner'] for tag in sublist)

    pos_mapping = {tag: idx for idx, tag in enumerate(sorted(pos_set))}
    ner_mapping = {tag: idx for idx, tag in enumerate(sorted(ner_set))}

    df['pos_tag_id'] = df['pos'].apply(lambda tags: [pos_mapping[tag] for tag in tags])
    df['ner_tag_id'] = df['ner'].apply(lambda tags: [ner_mapping[tag] for tag in tags])

    return df, pos_mapping, ner_mapping

In [7]:
def train_fn(data_loader, model, optimizer, device, scheduler):
    model.train()
    total_loss = 0
    for data in tqdm(data_loader, total=len(data_loader)):
        for k, v in data.items():
            data[k] = v.to(device)
        optimizer.zero_grad()
        _, _, loss = model(**data)
        loss.backward()
        optimizer.step()
        scheduler.step()
        total_loss += loss.item()
    return total_loss / len(data_loader)

In [8]:
def val_fn(data_loader, model, device):
    model.eval()
    losses = []
    all_preds_pos = []
    all_targets_pos = []
    all_preds_ner = []
    all_targets_ner = []

    with torch.no_grad():
        for batch in data_loader:
            ids = batch['ids'].to(device)
            mask = batch['mask'].to(device)
            token_type_ids = batch['token_type_ids'].to(device)
            target_pos = batch['target_pos'].to(device)
            target_ner = batch['target_ner'].to(device)
            
            pos, ner, loss = model(ids, mask, token_type_ids, target_pos, target_ner)
            losses.append(loss.item())
            
            # Collect all predictions and targets
            all_preds_pos.extend(pos.argmax(-1).detach().cpu().numpy().flatten())
            all_targets_pos.extend(target_pos.detach().cpu().numpy().flatten())
            all_preds_ner.extend(ner.argmax(-1).detach().cpu().numpy().flatten())
            all_targets_ner.extend(target_ner.detach().cpu().numpy().flatten())

    avg_loss = np.mean(losses)
    
    # Calculate metrics
    accuracy_pos = accuracy_score(all_targets_pos, all_preds_pos)
    accuracy_ner = accuracy_score(all_targets_ner, all_preds_ner)
    recall_pos = recall_score(all_targets_pos, all_preds_pos, average='weighted')
    recall_ner = recall_score(all_targets_ner, all_preds_ner, average='weighted')
    precision_pos = precision_score(all_targets_pos, all_preds_pos, average='weighted')
    precision_ner = precision_score(all_targets_ner, all_preds_ner, average='weighted')
    f1_pos = f1_score(all_targets_pos, all_preds_pos, average='weighted')
    f1_ner = f1_score(all_targets_ner, all_preds_ner, average='weighted')

    return avg_loss, accuracy_pos, accuracy_ner, recall_pos, recall_ner, precision_pos, precision_ner, f1_pos, f1_ner

In [9]:
def get_hyperparameters(model, ff=True, weight_decay=0.01):
    if ff:
        param_optimizer = list(model.named_parameters())
        no_decay = ["bias", "LayerNorm.weight"]
        optimizer_grouped_parameters = [
            {"params": [p for n, p in param_optimizer if not any(nd in n for nd in no_decay)], "weight_decay": weight_decay},
            {"params": [p for n, p in param_optimizer if any(nd in n for nd in no_decay)], "weight_decay": 0.0},
        ]
        optimizer = torch.optim.AdamW(optimizer_grouped_parameters, lr=5e-5)
        return optimizer

In [10]:
# Prepare the dataset and DataLoader
file_path = 'Data/data.tsv'
df, pos_mapping, ner_mapping = parse_dataset(file_path)

df

,words,pos,ner,pos_tag_id,ner_tag_id
0,"[শনিবার, (২৭, আগস্ট), রাতে, পটুয়াখালী, সদর, থ...","[NNP, PUNCT, NNP, NNC, NNP, NNC, NNC, ADJ, NNC...","[B-D&T, B-OTH, B-D&T, B-D&T, B-GPE, I-GPE, I-G...","[6, 11, 6, 5, 6, 5, 5, 0, 5, 11, 6, 6, 3, 5, 0...","[0, 7, 0, 0, 2, 13, 13, 8, 18, 7, 8, 18, 7, 7,..."
1,"[বায়ুদূষণ, ও, স্মার্ট, ফোন, ছেলেমেয়ে, উভয়ের, প...","[NNC, CONJ, NNC, NNC, NNC, PRO, NNC, NNC, NNC,...","[B-OTH, B-OTH, B-OTH, B-OTH, B-PER, B-OTH, B-O...","[5, 2, 5, 5, 5, 10, 5, 5, 5, 14, 13]","[7, 7, 7, 7, 8, 7, 7, 7, 7, 7, 7]"
2,"[ছাত্র, রাজনীতির, বর্তমান, অবস্থার, শুরু, হয়ে...","[NNC, NNC, ADJ, NNC, NNC, VF, NNC, NNP, NNC, PP]","[B-OTH, B-OTH, B-OTH, B-OTH, B-OTH, B-OTH, B-P...","[5, 5, 0, 5, 5, 13, 5, 6, 5, 9]","[7, 7, 7, 7, 7, 7, 8, 8, 7, 7]"
3,"[শাকিল, রাজধানীর, ৩০০, ফিট,, দিয়াবাড়ি, ও, পূ...","[NNP, NNC, QF, NNC, NNP, CONJ, NNP, NNC, NNC, ...","[B-PER, B-OTH, B-LOC, I-LOC, B-LOC, B-OTH, B-L...","[6, 5, 12, 5, 6, 2, 6, 5, 5, 9, 5, 14, 13, 9, ...","[8, 7, 3, 14, 3, 7, 3, 7, 7, 7, 8, 7, 7, 7, 7, 6]"
4,"[সম্প্রতি, ক্লাবের, নবীন, ব্যবস্থাপনা, প্রশিক্...","[ADV, NNC, ADJ, NNC, NNC, CONJ, NNC, NNC, PP, ...","[B-OTH, B-ORG, B-OTH, B-OTH, B-PER, B-OTH, B-P...","[1, 5, 0, 5, 5, 2, 5, 5, 9, 0, 13, 5, 5]","[7, 6, 7, 7, 8, 7, 8, 8, 7, 7, 7, 1, 12]"
...,...,...,...,...,...
6997,"[বিপদ:, প্লাস্টিকের, ব্যাগগুলি, বিপজ্জনক, হতে,...","[NNC, NNC, NNC, ADJ, VNF, VF]","[B-OTH, B-OTH, B-OTH, B-OTH, B-OTH, B-OTH]","[5, 5, 5, 0, 14, 13]","[7, 7, 7, 7, 7, 7]"
6998,"[তবে, ভুয়া, বিজ্ঞাপন, বন্ধে, ফেসবুকে, সঙ্গে, ...","[CONJ, ADJ, NNC, NNC, NNP, PP, NNC, VF, NNP]","[B-OTH, B-OTH, B-OTH, B-OTH, B-ORG, B-OTH, B-O...","[2, 0, 5, 5, 6, 9, 5, 13, 6]","[7, 7, 7, 7, 6, 7, 7, 7, 8]"
6999,"[১., ২০১৮, সালে, ঘোষিত, সরকারি, চাকরিতে, কোটা,...","[OTH, QF, NNC, ADJ, ADJ, NNC, NNC, NNC, ADJ, C...","[B-NUM, B-D&T, I-D&T, B-OTH, B-OTH, B-OTH, B-O...","[7, 12, 5, 0, 0, 5, 5, 5, 0, 2, 0, 5, 5, 5, 14...","[5, 0, 11, 7, 7, 7, 7, 7, 7, 7, 7, 7, 7, 7, 7, 7]"
7000,"['কয়েদীরা, দিনপঞ্জীর, মতো, দেয়ালে, নিজেদের, স্...","[PUNCT, NNC, PP, NNC, PRO, NNC, VNF, VF]","[B-OTH, B-OTH, B-OTH, B-OTH, B-OTH, B-OTH, B-O...","[11, 5, 9, 5, 10, 5, 14, 13]","[7, 7, 7, 7, 7, 7, 7, 7]"


In [11]:
texts = df['words'].values
pos_tags = df['pos_tag_id'].values
ner_tags = df['ner_tag_id'].values

In [12]:
texts = df['words'].values
pos_tags = df['pos_tag_id'].values
ner_tags = df['ner_tag_id'].values
train_texts, val_texts, train_pos, val_pos, train_ner, val_ner = train_test_split(
    texts, pos_tags, ner_tags, test_size=0.2, random_state=42
)

In [13]:
train_dataset = CustomDataset(train_texts, train_pos, train_ner)
val_dataset = CustomDataset(val_texts, val_pos, val_ner)

train_data_loader = DataLoader(train_dataset, batch_size=Config.TRAIN_BATCH_SIZE, shuffle=True)
val_data_loader = DataLoader(val_dataset, batch_size=Config.VAL_BATCH_SIZE, shuffle=False)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = NERPOSModel(num_pos=len(pos_mapping), num_ner=len(ner_mapping)).to(device)

model.safetensors:   0%|          | 0.00/1.12G [00:00<?, ?B/s]

In [14]:
optimizer = get_hyperparameters(model)
scheduler = get_linear_schedule_with_warmup(
    optimizer, num_warmup_steps=0, num_training_steps=len(train_data_loader) * Config.EPOCHS
)

print(model)

NERPOSModel(
  (bert): XLMRobertaModel(
    (embeddings): XLMRobertaEmbeddings(
      (word_embeddings): Embedding(250002, 768, padding_idx=1)
      (position_embeddings): Embedding(514, 768, padding_idx=1)
      (token_type_embeddings): Embedding(1, 768)
      (LayerNorm): LayerNorm((768,), eps=1e-05, elementwise_affine=True)
      (dropout): Dropout(p=0.1, inplace=False)
    )
    (encoder): XLMRobertaEncoder(
      (layer): ModuleList(
        (0-11): 12 x XLMRobertaLayer(
          (attention): XLMRobertaAttention(
            (self): XLMRobertaSelfAttention(
              (query): Linear(in_features=768, out_features=768, bias=True)
              (key): Linear(in_features=768, out_features=768, bias=True)
              (value): Linear(in_features=768, out_features=768, bias=True)
              (dropout): Dropout(p=0.1, inplace=False)
            )
            (output): XLMRobertaSelfOutput(
              (dense): Linear(in_features=768, out_features=768, bias=True)
              (

In [15]:
for epoch in range(Config.EPOCHS):
    print(f"Epoch {epoch + 1}/{Config.EPOCHS}")
    train_loss = train_fn(train_data_loader, model, optimizer, device, scheduler)
    val_loss, val_acc_pos, val_acc_ner, val_rec_pos, val_rec_ner, val_prec_pos, val_prec_ner, val_f1_pos, val_f1_ner = val_fn(val_data_loader, model, device)
    
    print(f"Train Loss: {train_loss:.4f}")
    print(f"Validation Loss: {val_loss:.4f}")
    print(f"Validation POS Accuracy: {val_acc_pos:.4f}, NER Accuracy: {val_acc_ner:.4f}")
    print(f"Validation POS Recall: {val_rec_pos:.4f}, NER Recall: {val_rec_ner:.4f}")
    print(f"Validation POS Precision: {val_prec_pos:.4f}, NER Precision: {val_prec_ner:.4f}")
    print(f"Validation POS F1 Score: {val_f1_pos:.4f}, NER F1 Score: {val_f1_ner:.4f}")

Epoch 1/15


100%|██████████| 351/351 [01:06<00:00,  5.32it/s]
/opt/conda/lib/python3.10/site-packages/sklearn/metrics/_classification.py:1344: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, msg_start, len(result))
/opt/conda/lib/python3.10/site-packages/sklearn/metrics/_classification.py:1344: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, msg_start, len(result))


Train Loss: 0.7277
Validation Loss: 0.3172
Validation POS Accuracy: 0.8501, NER Accuracy: 0.8491
Validation POS Recall: 0.8501, NER Recall: 0.8491
Validation POS Precision: 0.9224, NER Precision: 0.9159
Validation POS F1 Score: 0.8710, NER F1 Score: 0.8663
Epoch 2/15


100%|██████████| 351/351 [01:07<00:00,  5.17it/s]
/opt/conda/lib/python3.10/site-packages/sklearn/metrics/_classification.py:1344: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, msg_start, len(result))
/opt/conda/lib/python3.10/site-packages/sklearn/metrics/_classification.py:1344: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, msg_start, len(result))


Train Loss: 0.3569
Validation Loss: 0.2543
Validation POS Accuracy: 0.8641, NER Accuracy: 0.8501
Validation POS Recall: 0.8641, NER Recall: 0.8501
Validation POS Precision: 0.9277, NER Precision: 0.9186
Validation POS F1 Score: 0.8837, NER F1 Score: 0.8676
Epoch 3/15


...

Train Loss: 0.1179
Validation Loss: 0.2118
Validation POS Accuracy: 0.8710, NER Accuracy: 0.8686
Validation POS Recall: 0.8710, NER Recall: 0.8686
Validation POS Precision: 0.9334, NER Precision: 0.9232
Validation POS F1 Score: 0.8898, NER F1 Score: 0.8824
Epoch 14/15


100%|██████████| 351/351 [01:08<00:00,  5.15it/s]


Train Loss: 0.1111
Validation Loss: 0.2134
Validation POS Accuracy: 0.8721, NER Accuracy: 0.8710
Validation POS Recall: 0.8721, NER Recall: 0.8710
Validation POS Precision: 0.9336, NER Precision: 0.9244
Validation POS F1 Score: 0.8908, NER F1 Score: 0.8845
Epoch 15/15


100%|██████████| 351/351 [01:08<00:00,  5.15it/s]


Train Loss: 0.1099
Validation Loss: 0.2136
Validation POS Accuracy: 0.8731, NER Accuracy: 0.8704
Validation POS Recall: 0.8731, NER Recall: 0.8704
Validation POS Precision: 0.9337, NER Precision: 0.9236
Validation POS F1 Score: 0.8914, NER F1 Score: 0.8838


In [16]:
def show_example_from_validation(val_dataset, model, tokenizer, pos_mapping, ner_mapping):
    # Get a single example from the validation dataset
    example = val_dataset[0]  # Taking the first example in the validation set
    ids = example['ids'].unsqueeze(0).to(device)
    mask = example['mask'].unsqueeze(0).to(device)
    token_type_ids = example['token_type_ids'].unsqueeze(0).to(device)
    target_pos = example['target_pos'].unsqueeze(0).to(device)
    target_ner = example['target_ner'].unsqueeze(0).to(device)

    model.eval()
    with torch.no_grad():
        pos_logits, ner_logits, _ = model(ids, mask, token_type_ids, target_pos, target_ner)
    
    # Get the predicted tags
    pos_preds = torch.argmax(pos_logits, dim=2).cpu().numpy()[0]
    ner_preds = torch.argmax(ner_logits, dim=2).cpu().numpy()[0]
    
    # Get the actual tags
    actual_pos = target_pos.cpu().numpy()[0]
    actual_ner = target_ner.cpu().numpy()[0]
    
    # Convert IDs to tokens
    tokens = tokenizer.convert_ids_to_tokens(ids[0].cpu().numpy())
    
    # Convert predictions and actual values to tag names
    predicted_pos_tags = [list(pos_mapping.keys())[list(pos_mapping.values()).index(tag)] for tag in pos_preds if tag != 0]
    predicted_ner_tags = [list(ner_mapping.keys())[list(ner_mapping.values()).index(tag)] for tag in ner_preds if tag != 0]

    actual_pos_tags = [list(pos_mapping.keys())[list(pos_mapping.values()).index(tag)] for tag in actual_pos if tag != 0]
    actual_ner_tags = [list(ner_mapping.keys())[list(ner_mapping.values()).index(tag)] for tag in actual_ner if tag != 0]
    
    print("Tokens: ", tokens)
    print("Predicted POS tags: ", predicted_pos_tags)
    print("Actual POS tags: ", actual_pos_tags)
    print("Predicted NER tags: ", predicted_ner_tags)
    print("Actual NER tags: ", actual_ner_tags)

# Call the function to show an example from the validation set
show_example_from_validation(val_dataset, model, Config.TOKENIZER, pos_mapping, ner_mapping)

Tokens:  ['<s>', '▁মানব', 'স', 'ভ', '্য', 'তার', '▁বি', 'স্', 'ময়', 'কর', '▁বি', 'কা', 'শে', '▁আধুনিক', '▁বিজ্ঞান', '▁যে', '▁অন', 'ন্য', '▁ভূমিকা', '▁পালন', '▁করছে', '▁তার', '</s>', '<pad>', '<pad>', '<pad>', '<pad>', '<pad>', '<pad>', '<pad>', '<pad>', '<pad>', '<pad>', '<pad>', '<pad>', '<pad>', '<pad>', '<pad>', '<pad>', '<pad>', '<pad>', '<pad>', '<pad>', '<pad>', '<pad>', '<pad>', '<pad>', '<pad>', '<pad>', '<pad>', '<pad>', '<pad>', '<pad>', '<pad>', '<pad>', '<pad>', '<pad>', '<pad>', '<pad>', '<pad>', '<pad>', '<pad>', '<pad>', '<pad>', '<pad>', '<pad>', '<pad>', '<pad>', '<pad>', '<pad>', '<pad>', '<pad>', '<pad>', '<pad>', '<pad>', '<pad>', '<pad>', '<pad>', '<pad>', '<pad>', '<pad>', '<pad>', '<pad>', '<pad>', '<pad>', '<pad>', '<pad>', '<pad>', '<pad>', '<pad>', '<pad>', '<pad>', '<pad>', '<pad>', '<pad>', '<pad>', '<pad>', '<pad>', '<pad>', '<pad>', '<pad>', '<pad>', '<pad>', '<pad>', '<pad>', '<pad>', '<pad>', '<pad>', '<pad>', '<pad>', '<pad>', '<pad>', '<pad>', '<pad>'

In [19]:
def prediction(test_sentence, model, tokenizer, pos_mapping, ner_mapping):
    # Tokenize the input sentence
    tokens = tokenizer(test_sentence, return_tensors="pt", padding=True, truncation=True, max_length=Config.MAX_LEN)
    input_ids = tokens['input_ids'].to(device)
    attention_mask = tokens['attention_mask'].to(device)
    
    # Handle token_type_ids if present
    if 'token_type_ids' in tokens:
        token_type_ids = tokens['token_type_ids'].to(device)
    else:
        token_type_ids = None

    model.eval()
    with torch.no_grad():
        pos_logits, ner_logits, _ = model(input_ids, attention_mask, token_type_ids, None, None)
    
    # Get the predicted tags
    pos_preds = torch.argmax(pos_logits, dim=2).cpu().numpy()[0]
    ner_preds = torch.argmax(ner_logits, dim=2).cpu().numpy()[0]

    # Convert ids to tokens and skip special tokens (like [CLS] and [SEP])
    tokens = tokenizer.convert_ids_to_tokens(input_ids[0])[1:-1]
    pos_preds = pos_preds[1:-1]  # Skip [CLS] and [SEP]
    ner_preds = ner_preds[1:-1]  # Skip [CLS] and [SEP]

    # Re-align predictions with the original words
    pos_tags = []
    ner_tags = []
    current_word_idx = 0

    for token, pos_pred, ner_pred in zip(tokens, pos_preds, ner_preds):
        if token.startswith("▁"):  # Word boundary in sentencepiece-based tokenizers like XLM-Roberta
            current_word_idx += 1

        # Only keep the prediction for the first token of each word
        if len(pos_tags) < current_word_idx:
            pos_tags.append(list(pos_mapping.keys())[list(pos_mapping.values()).index(pos_pred)])
            ner_tags.append(list(ner_mapping.keys())[list(ner_mapping.values()).index(ner_pred)])
    
    # Align the tags with the original sentence
    return list(zip(test_sentence.split(), pos_tags, ner_tags))

In [20]:
test_sentence = "বাংলাদেশের ইতিহাসে বাংলাদেশ কখনওই রাষ্ট্র হিসেবে একটি সফল হতে পারেনি"
result = prediction(test_sentence, model, Config.TOKENIZER, pos_mapping, ner_mapping)
print(result)

[('বাংলাদেশের', 'NNP', 'B-GPE'), ('ইতিহাসে', 'NNC', 'B-OTH'), ('বাংলাদেশ', 'NNP', 'B-GPE'), ('কখনওই', 'ADV', 'B-OTH'), ('রাষ্ট্র', 'NNC', 'B-OTH'), ('হিসেবে', 'PP', 'B-OTH'), ('একটি', 'QF', 'B-NUM'), ('সফল', 'ADJ', 'B-OTH'), ('হতে', 'VNF', 'B-OTH'), ('পারেনি', 'VF', 'B-OTH')]


In [21]:
# Save the model
MODEL_PATH = "ner_pos_model_xlm_roberta.pt"
torch.save(model.state_dict(), MODEL_PATH)

In [22]:
# Test the function with a sample sentence
test_sentence_2 = "শনিবার (২৭ আগস্ট) রাতে পটুয়াখালী সদর থানার ভারপ্রাপ্ত কর্মকর্তা (ওসি) মো. মনিরুজ্জামান এ তথ্য নিশ্চিত করেছেন।"
result = prediction(test_sentence_2, model, Config.TOKENIZER, pos_mapping, ner_mapping)
print(result)

[('শনিবার', 'NNP', 'B-D&T'), ('(২৭', 'PUNCT', 'B-OTH'), ('আগস্ট)', 'NNP', 'B-D&T'), ('রাতে', 'NNC', 'B-D&T'), ('পটুয়াখালী', 'NNP', 'B-GPE'), ('সদর', 'NNP', 'I-ORG'), ('থানার', 'NNC', 'I-ORG'), ('ভারপ্রাপ্ত', 'ADJ', 'B-PER'), ('কর্মকর্তা', 'NNC', 'I-PER'), ('(ওসি)', 'PUNCT', 'B-OTH'), ('মো.', 'NNP', 'B-PER'), ('মনিরুজ্জামান', 'NNP', 'I-PER'), ('এ', 'DET', 'B-OTH'), ('তথ্য', 'NNC', 'B-OTH'), ('নিশ্চিত', 'ADJ', 'B-OTH'), ('করেছেন।', 'VF', 'B-OTH')]
